In [1]:
import gymnasium as gym
from gymnasium import spaces

class TailRiskExplorerEnv(gym.Env):
    def __init__(self, world_model, baseline_tensor, env_feature_indices):
        super(TailRiskExplorerEnv, self).__init__()
        self.model = world_model
        self.dna = baseline_tensor
        self.env_indices = env_feature_indices
        
        # Action Space: Perturb up to 3 macro variables continuously (-5.0 to +5.0)
        self.action_space = spaces.Box(low=-5.0, high=5.0, shape=(len(env_feature_indices),))
        
        # Observation Space: The current predicted risk (0.0 to 1.0)
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(1,))

    def step(self, action):
        # Apply RL-generated shocks to the macro features
        shocked_tensor = self.dna.clone()
        for i, (feat_name, idx) in enumerate(self.env_indices.items()):
            shocked_tensor[:, :, idx] += action[i]
            
        with torch.no_grad():
            risk_score = self.model(shocked_tensor).mean().item()
            
        # Reward function targeting Tail Risks > 15%
        if risk_score > 0.15:
            reward = 10.0 + (risk_score * 10) # Bonus for finding extreme risks
        else:
            reward = -1.0 # Penalize boring, normal scenarios
            
        # Terminate episode if a black swan is found
        terminated = bool(risk_score > 0.15)
        
        return [risk_score], reward, terminated, False, {}